# Publish a private-network Foundry agent to M365 & Teams — raw REST walkthrough

This notebook makes **every HTTP request explicit** — you see each method, URL, header, and JSON body, and the raw response. Nothing is hidden behind helper functions.

Microsoft 365 **does not support private connectivity** to agents; it requires the agent's **Activity Protocol** endpoint to be publicly routable. So for a locked-down (private-network) project we open a scoped, **source-IP-filtered** public exception on *only* the Activity Protocol route (service-managed by Foundry) and route Teams/Copilot messages through an Azure **Bot Service**. Everything else stays private.

**The 5 calls:**
1. `GET  /agents/{name}` — read the agent identity (`instance_identity.client_id`) + current endpoint config
2. `az deployment group create` — create the Azure **Bot Service** (PNA disabled + Teams channel)
3. `PATCH /agents/{name}` — set `activity.enable_m365_public_endpoint: true` *(locked-down projects only)*
4. `POST /agents/{name}/microsoft365/publish` — publish to M365 → returns `titleId`
5. Verify in Teams / M365 Copilot

Reference: [Publish agents to M365 & Teams via REST](https://learn.microsoft.com/azure/foundry/agents/how-to/publish-copilot-virtual-network?view=foundry)

## Prerequisites

- **Roles:** `Foundry User` on the project + `Azure Bot Service Contributor` (or Contributor/Owner) on the resource group.
- **Run from inside the VNet** (VM / VPN / ExpressRoute) so these management calls can reach the project's private endpoint.
- A **persistent, published** agent version (stable name/version).

Run the setup cell once (safe to re-run):

In [ ]:
%pip install -q azure-identity requests
!az provider register --namespace Microsoft.BotService
!az login
!az account show --query "{subscription:name, id:id, tenant:tenantId}" -o table

## Configure

Fill these in. `DRY_RUN = True` means the mutating cells (Bot Service, PATCH, publish) **print the exact request but do not send it**. Flip to `False` to actually run them.

In [ ]:
endpoint        = "https://<res>.services.ai.azure.com/api/projects/<proj>"  # project endpoint
resource_group  = "<your-resource-group>"   # RG that contains the Foundry resource
agent_name      = "<your-agent-name>"        # persistent, published agent

# Bot Service to create (Step 2)
bot_name        = "<new-bot-name>"

# M365 publish metadata (Step 4) — all user-visible, so no secrets
agent_display_name    = "<Display Name>"
publish_scope         = "Shared"             # Shared | Personal | Tenant
app_version           = "1.0.0"              # increment to re-publish
short_description     = "<short description>"
full_description      = "<full description>"
developer_name        = "<developer name>"
developer_website_url = "https://<developer-website>"
privacy_url           = "https://<privacy-url>"
terms_of_use_url      = "https://<terms-of-use-url>"

DRY_RUN = True   # <-- set False to actually create the bot / PATCH / publish

endpoint = endpoint.rstrip("/")
API_VERSION          = "v1"                    # management + publish API
ACTIVITY_API_VERSION = "2025-05-15-preview"    # activity protocol route

# Scope -> Bot Service authorization scheme (Step 3 must match Step 4's scope)
scheme = "BotServiceTenant" if publish_scope == "Tenant" else "BotServiceRbac"
print(f"agent={agent_name}  scope={publish_scope}  scheme={scheme}  DRY_RUN={DRY_RUN}")

## Acquire a bearer token

All REST calls authenticate with an Entra bearer token for the `https://ai.azure.com` audience, taken from your `az login` session. The `Authorization` header on every request below is literally `Bearer <this token>`.

In [ ]:
import os, json, subprocess
from urllib.parse import urlparse
import requests
from azure.identity import AzureCliCredential

credential = AzureCliCredential()
access_token = credential.get_token("https://ai.azure.com/.default").token
headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
print(f"Token acquired ({len(access_token)} chars). Header: Authorization: Bearer <token>")

## Step 1 — `GET /agents/{name}`: read the agent identity + current config

The Bot Service and the M365 publish need the agent's **identity client ID** (`instance_identity.client_id`). We also print the current protocol/auth config so you can see what Step 3's PATCH will change (and confirm what it must re-send, because the PATCH **replaces** those blocks).

In [ ]:
url = f"{endpoint}/agents/{agent_name}?api-version={API_VERSION}"
print("GET", url, "\n")

resp = requests.get(url, headers=headers, timeout=60)
resp.raise_for_status()
agent = resp.json()

client_id = agent["instance_identity"]["client_id"]

# tenant ID for the Bot Service (msaAppTenantId)
tenant_id = subprocess.run(
    ["az", "account", "show", "--query", "tenantId", "-o", "tsv"],
    capture_output=True, text=True, shell=(os.name == "nt"),
).stdout.strip()

ep_cfg  = agent.get("agent_endpoint", {}) or {}
protos  = list((ep_cfg.get("protocol_configuration") or {}).keys())
schemes = [s.get("type") for s in ep_cfg.get("authorization_schemes", []) or []]
activity = (ep_cfg.get("protocol_configuration") or {}).get("activity", {}) or {}

print("client_id (agent identity):", client_id)
print("tenant_id:                 ", tenant_id)
print("current protocols:         ", protos)
print("current auth schemes:      ", schemes)
print("enable_m365_public_endpoint:", activity.get("enable_m365_public_endpoint", False))

## Step 2 — Create the Azure Bot Service

The Bot Service is the trusted proxy that relays Teams/Copilot messages to the agent's private Activity Protocol endpoint. We deploy it from [`scripts/bot_service.bicep`](scripts/bot_service.bicep) — printed below so you can see exactly what is created (note `publicNetworkAccess: 'Disabled'` and the `MsTeamsChannel`).

The `endpoint` parameter is the agent's Activity Protocol route:
`{endpoint}/agents/{name}/endpoint/protocols/activityProtocol?api-version={ACTIVITY_API_VERSION}`

The exact `az deployment group create` command is printed before it runs.

In [ ]:
from pathlib import Path

bicep_path = Path("scripts/bot_service.bicep")
print("----- bot_service.bicep -----")
print(bicep_path.read_text())
print("-----------------------------\n")

activity_endpoint = (
    f"{endpoint}/agents/{agent_name}/endpoint/protocols/activityProtocol"
    f"?api-version={ACTIVITY_API_VERSION}"
)

cmd = [
    "az", "deployment", "group", "create",
    "--resource-group", resource_group,
    "--template-file", str(bicep_path),
    "--parameters",
    f"botName={bot_name}",
    f"displayName={agent_display_name or agent_name}",
    f"msaAppId={client_id}",
    f"tenantId={tenant_id}",
    f"endpoint={activity_endpoint}",
    "--query", "properties.outputs.botServiceArmId.value", "-o", "tsv",
]
print("RUN:", " ".join(cmd), "\n")

if DRY_RUN:
    bot_arm_id = "<dry-run: bot not created>"
    print("[dry-run] Bot Service not deployed.")
else:
    result = subprocess.run(cmd, capture_output=True, text=True, shell=(os.name == "nt"))
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    bot_arm_id = result.stdout.strip()

print("botServiceArmId:", bot_arm_id)

### (Optional) Reuse an existing Bot Service instead of creating one

If a Bot Service is already wired to this agent, skip Step 2 and just set its ARM ID. This lists every Bot Service in the subscription and flags one whose `msaAppId` matches the agent identity or whose `endpoint` targets this agent.

In [ ]:
sub = subprocess.run(["az", "account", "show", "--query", "id", "-o", "tsv"],
                     capture_output=True, text=True, shell=(os.name == "nt")).stdout.strip()
list_url = (f"https://management.azure.com/subscriptions/{sub}"
            f"/providers/Microsoft.BotService/botServices?api-version=2022-09-15")
out = subprocess.run(["az", "rest", "--method", "get", "--url", list_url, "--query", "value", "-o", "json"],
                     capture_output=True, text=True, shell=(os.name == "nt")).stdout
needle = f"/agents/{agent_name}/"
for b in json.loads(out or "[]"):
    p = b.get("properties", {}) or {}
    if p.get("msaAppId") == client_id or needle in (p.get("endpoint") or ""):
        print("MATCH:", b["id"])
        print("  msaAppId:", p.get("msaAppId"), "| endpoint:", p.get("endpoint"))
        # bot_arm_id = b["id"]   # <- uncomment to reuse this bot instead of creating one
    else:
        print("other bot:", b.get("id"))

## Step 3 — `PATCH /agents/{name}`: enable the public Activity Protocol endpoint

**Only needed when the project has public network access disabled/restricted.** First we read the Foundry account's `publicNetworkAccess`; if it's unrestricted, this step is skipped (Foundry's one-click publish would already work).

The PATCH uses `Content-Type: application/merge-patch+json` and **replaces** `protocol_configuration` and `authorization_schemes` — so we re-send `responses`, `Entra`, and the scope-matched Bot Service scheme, and add `activity.enable_m365_public_endpoint: true`.

> The source-IP filtering itself is **service-managed by Foundry** — this boolean is the only knob; Microsoft maintains the allowed Bot Service / M365 source ranges. Auth (`Entra` / `BotServiceRbac` / `BotServiceTenant`) still applies on top.

In [ ]:
# Read public network access on the parent Foundry (AI Services) account.
account = (urlparse(endpoint).hostname or "").split(".")[0]
show = subprocess.run(
    ["az", "cognitiveservices", "account", "show",
     "--name", account, "--resource-group", resource_group,
     "--query", "{pna:properties.publicNetworkAccess,ip:properties.networkAcls.ipRules,"
                "vnet:properties.networkAcls.virtualNetworkRules}", "-o", "json"],
    capture_output=True, text=True, shell=(os.name == "nt"),
)
posture = json.loads(show.stdout) if show.returncode == 0 and show.stdout.strip() else {}
pna = posture.get("pna")
restricted = (pna == "Disabled") or bool(posture.get("ip")) or bool(posture.get("vnet"))
print(f"publicNetworkAccess={pna!r}  restricted={restricted}  (PATCH required: {restricted})\n")

patch_url = f"{endpoint}/agents/{agent_name}?api-version={API_VERSION}"
patch_body = {
    "agent_endpoint": {
        "protocol_configuration": {
            "responses": {},
            "activity": {"enable_m365_public_endpoint": True},
        },
        "authorization_schemes": [
            {"type": "Entra"},
            {"type": scheme},
        ],
    }
}
patch_headers = {**headers, "Content-Type": "application/merge-patch+json"}

FORCE_PATCH = False  # set True to PATCH even on an unrestricted (public) project
if restricted or FORCE_PATCH:
    print("PATCH", patch_url)
    print("Content-Type: application/merge-patch+json")
    print(json.dumps(patch_body, indent=2), "\n")
    if DRY_RUN:
        print("[dry-run] PATCH not sent.")
    else:
        r = requests.patch(patch_url, headers=patch_headers, json=patch_body, timeout=60)
        r.raise_for_status()
        print("PATCH ok:", r.status_code)
else:
    print("Skipping PATCH — project allows public access, exception not required.")

## Step 4 — `POST /microsoft365/publish`: publish to Microsoft 365

The full request body is printed below. `botServiceArmId` links the publish to the Bot Service from Step 2. A successful response returns a `titleId`.

- **Scope ↔ auth:** `Shared`/`Personal` → `BotServiceRbac` (you only, share by link, no admin approval); `Tenant` → `BotServiceTenant` (whole tenant, after M365 admin approval).
- **Re-publishing** requires a **new `appVersion`** (digits/periods only, cannot start with `0`) — a duplicate returns a *version already exists* error.
- **No secrets** in any metadata field — they are user-visible.

In [ ]:
publish_url = f"{endpoint}/agents/{agent_name}/microsoft365/publish?api-version={API_VERSION}"
publish_body = {
    "agentDisplayName": agent_display_name or agent_name,
    "botServiceArmId": bot_arm_id,
    "publishScope": publish_scope,
    "publishAsAutopilot": False,
    "appVersion": app_version,
    "shortDescription": short_description,
    "fullDescription": full_description,
    "developerName": developer_name,
    "developerWebsiteUrl": developer_website_url,
    "privacyUrl": privacy_url,
    "termsOfUseUrl": terms_of_use_url,
}
print("POST", publish_url)
print(json.dumps(publish_body, indent=2), "\n")

if DRY_RUN:
    print("[dry-run] publish not sent.")
elif str(bot_arm_id).startswith("<"):
    print("No real botServiceArmId — run Step 2 (or set bot_arm_id) before publishing.")
else:
    r = requests.post(publish_url, headers=headers, json=publish_body, timeout=60)
    r.raise_for_status()
    print("Published. titleId =", (r.json() or {}).get("titleId"))

## Step 5 — Verify

1. Open the M365 / Teams agent store: `Shared` → **Your agents**; `Tenant` → **Built by your org** (after admin approval).
2. Start a conversation and send a message.
3. Confirm the agent replies — that validates end-to-end channel delivery.

**What stayed private:** only inbound Teams/M365 message delivery uses the source-IP-filtered public route; the agent's Foundry→Fabric query still traverses Private Link, and outbound/egress is unchanged. **Limitations:** no file uploads / image generation in M365 (works in Teams); no streaming responses or citations for published agents.